# V6 Reviewer Experiments L4

Paket eksperimen CYBERBULLY untuk menutup persoalan evaluasi sebelum penulisan v6.

**Isi:** perbaikan split URL • audit pasangan video • pelatihan ulang cabang video • ablasi coverage • pembanding HEALNet 2024 dan SigLIP 2 2025 • tiga seed • statistik per kelompok • ekspor hasil.

**Cara pakai:** unggah/buka notebook di Google Colab, pilih L4 GPU, pastikan folder `CYBERBULLY ` ada sebagai shortcut di MyDrive, lalu jalankan semua sel. Hasil disimpan dalam direktori baru `results/v6_reviewer_experiments_20261020_v1/`. Jika koneksi terputus, jalankan ulang dengan konfigurasi sama.

**Sumber primer:** [HEALNet resmi](https://github.com/konst-int-i/healnet), [SigLIP 2 resmi](https://huggingface.co/google/siglip2-base-patch16-224), [IndoBERTweet resmi](https://huggingface.co/indolem/indobertweet-base-uncased).

**Batas:** eksperimen ini tidak menjamin kenaikan skor atau penerimaan jurnal. Kesesuaian label, bukti anotasi dan definisi child-directed tetap memerlukan verifikasi manusia. Tidak mengubah label atau memasukkan angka hasil yang belum dihasilkan.


## 1 Konfigurasi L4 dan lokasi proyek

Jalankan berurutan atau **Runtime → Run all**. Root sengaja bernama `CYBERBULLY ` dengan satu spasi terakhir. Aktifkan L4 GPU sebelum mulai. Default menjalankan paket lengkap dan tiga seed video; pelatihan membutuhkan waktu dan compute units.

Konfigurasi dikunci sejak preflight. Untuk melanjutkan sesi yang terputus, gunakan notebook dan konfigurasi yang sama. Untuk eksperimen berbeda, ganti `run_directory`. Checkpoint optimizer disimpan per epoch dan dibuang setelah output tugas selesai; best weights serta prediksi dipertahankan. Penyimpanan bobot dan checkpoint beberapa model membutuhkan ruang Drive beberapa GB. Folder dan label lama tidak ditimpa.

Notebook dibuat dan diperiksa di lingkungan penulisan; **pelatihan L4 belum dieksekusi di sana**. Versi dependency dan commit model dicatat ketika Anda menjalankan notebook.


In [ ]:
import os, sys, subprocess, importlib.metadata

DEPENDENCIES = ['transformers==4.57.1','accelerate>=1.8,<2','emoji==2.14.1',
                'einops==0.8.1','sentencepiece>=0.2,<0.3','protobuf>=4.25,<7',
                'scikit-learn>=1.5,<2','pandas>=2.1,<3','scipy>=1.11,<2',
                'google-api-python-client>=2.140','safetensors>=0.5,<1']
subprocess.check_call([sys.executable,'-m','pip','install','-q',*DEPENDENCIES])
os.environ['TOKENIZERS_PARALLELISM']='false'
os.environ['CUBLAS_WORKSPACE_CONFIG']=':4096:8'

from pathlib import Path
from google.colab import drive
drive.mount('/content/drive',force_remount=False)

# Satu spasi sesudah CYBERBULLY disengaja. Jangan menghapusnya.
PROJECT_ROOT = Path('/content/drive/MyDrive/CYBERBULLY ')
if not (PROJECT_ROOT/'data/dataset_master_13307_labeled.csv').is_file():
    candidates=[p for p in Path('/content/drive/MyDrive').iterdir()
                if p.is_dir() and p.name.strip().casefold()=='cyberbully'
                and (p/'data/dataset_master_13307_labeled.csv').is_file()]
    if len(candidates)==1: PROJECT_ROOT=candidates[0]
    else: raise FileNotFoundError("Tambahkan shortcut folder CYBERBULLY ke MyDrive; root yang diharapkan berakhir satu spasi.")

CONFIG = {
 'protocol_version':'v6_reviewer_experiments_20261020_v1',
 'run_text':True, 'run_video':True, 'run_recent_baselines':True,
 'run_lopo_probe':True, 'run_missingness_stress':True,
 'text_models':{'indobert_p2':'indobenchmark/indobert-base-p2',
                'indobertweet':'indolem/indobertweet-base-uncased',
                'mbert':'google-bert/bert-base-multilingual-cased'},
 'text_seeds':[42], 'text_primary_extra_seeds':[2026,2027],
 'video_seeds':[42,2026,2027], 'primary_seed':42,
 'text_epochs':4,'video_text_epochs':4,'visual_epochs':5,
 'text_max_length':128,'video_max_length':192,
 'text_batch':16,'visual_batch':32,'accumulation':2,
 'head_epochs':30,'head_batch':64,'head_lr':0.001,
 'bootstrap_replicates':2000,'permutation_replicates':2000,
 'fusion_C':0.2,'base_audio_C':1.0,
 'threshold_grid':[round(.10+.025*i,3) for i in range(33)],
 'canonical_run_relative':'results/V6_FINAL_GPU_RUNS/canonical_efficientnet_b0_v2',
 'siglip2_model':'google/siglip2-base-patch16-224',
 'healnet_commit':'90459b65a3d4a4ef9fd405671c457b5a1163cc7d',
 'save_best_weights':True,
 'run_directory':'results/v6_reviewer_experiments_20261020_v1',
}
# Untuk sesi pendek, pilih [42] SEBELUM menjalankan. Konfigurasi terkunci setelah preflight.
# Ubah run_directory jika mengubah konfigurasi eksperimen; folder lama tidak ditimpa.
RUN = PROJECT_ROOT / CONFIG['run_directory']
assert CONFIG['primary_seed'] in CONFIG['video_seeds']
assert 'indobert_p2' in CONFIG['text_models'] and 42 in CONFIG['text_seeds']
LOCAL = Path('/content/v6_reviewer_experiments_cache')
RUN.mkdir(parents=True,exist_ok=True); LOCAL.mkdir(parents=True,exist_ok=True)
print('Project root:',repr(str(PROJECT_ROOT)))
print('Output:',RUN)



## 2 Runtime dan fungsi evaluasi

Metrik, pemilihan ambang, penyimpanan atomik, seed, dan pembentukan kelompok. GPU dipakai untuk pelatihan/inferensi; statistik berbasis kelompok dihitung di CPU.


In [ ]:
import json, hashlib, random, math, gc, time, re, html, shutil, zipfile, warnings
from collections import Counter,defaultdict
from urllib.parse import urlsplit,parse_qs,urlencode,unquote
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import Dataset,DataLoader
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline,FeatureUnion,Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (accuracy_score,f1_score,precision_recall_fscore_support,
 roc_auc_score,average_precision_score,confusion_matrix,brier_score_loss,cohen_kappa_score)
from scipy.stats import binomtest
from PIL import Image
from tqdm.auto import tqdm
from transformers import AutoTokenizer,AutoModel,AutoModelForSequenceClassification,AutoProcessor
from transformers import get_cosine_schedule_with_warmup
from huggingface_hub import HfApi

assert torch.cuda.is_available(),'Aktifkan Runtime > Change runtime type > L4 GPU, lalu jalankan ulang.'
DEVICE=torch.device('cuda')
GPU_NAME=torch.cuda.get_device_name(0)
USE_BF16=torch.cuda.is_bf16_supported()
AMP_DTYPE=torch.bfloat16 if USE_BF16 else torch.float16
torch.set_float32_matmul_precision('high')
torch.backends.cudnn.benchmark=False
torch.backends.cudnn.deterministic=True
torch.use_deterministic_algorithms(True,warn_only=True)
print('GPU:',GPU_NAME,'| AMP:',AMP_DTYPE)
if 'L4' not in GPU_NAME:print('GPU lain terdeteksi; kode tetap dapat berjalan dengan konfigurasi batch yang sama.')

def seed_all(seed):
    random.seed(seed);np.random.seed(seed);torch.manual_seed(seed);torch.cuda.manual_seed_all(seed)

def sha256(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda:f.read(4*1024*1024),b''):h.update(block)
    return h.hexdigest()

def model_revision(model_id):
    path=RUN/'MODEL_REVISIONS.json'
    registry=json.loads(path.read_text()) if path.is_file() else {}
    if model_id not in registry:
        registry[model_id]=HfApi().model_info(model_id).sha
        atomic_json(registry,path)
    return registry[model_id]

def atomic_json(value,path):
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    temp=path.with_name(path.name+'.partial')
    temp.write_text(json.dumps(value,indent=2,ensure_ascii=False,default=str));temp.replace(path)

def atomic_csv(frame,path):
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    temp=path.with_name(path.name+'.partial');frame.to_csv(temp,index=False);temp.replace(path)

def atomic_torch(value,path):
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    temp=path.with_name(path.name+'.partial');torch.save(value,temp);temp.replace(path)

def atomic_npz(path,**arrays):
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    temp=path.with_name(path.name+'.partial')
    with temp.open('wb') as stream:np.savez_compressed(stream,**arrays)
    temp.replace(path)

def cpu_state(model):return {k:v.detach().cpu().clone() for k,v in model.state_dict().items()}

def cleanup():
    gc.collect();torch.cuda.empty_cache()

def present(value):
    return not pd.isna(value) and str(value).strip().casefold() not in ('','-','nan','none','null','n/a')

def metrics(y,p,pred=None,threshold=.5):
    y=np.asarray(y,int);p=np.asarray(p,float)
    assert len(y)==len(p) and np.isfinite(p).all() and ((p>=0)&(p<=1)).all()
    pred=(p>=threshold).astype(int) if pred is None else np.asarray(pred,int)
    precision,recall,f1,support=precision_recall_fscore_support(y,pred,labels=[0,1],zero_division=0)
    return {'n':len(y),'accuracy':float(accuracy_score(y,pred)),
      'macro_f1':float(f1_score(y,pred,average='macro',zero_division=0)),
      'precision_macro':float(precision.mean()),'recall_macro':float(recall.mean()),
      'recall_nonbullying':float(recall[0]),'recall_bullying':float(recall[1]),
      'f1_nonbullying':float(f1[0]),'f1_bullying':float(f1[1]),
      'support_nonbullying':int(support[0]),'support_bullying':int(support[1]),
      'roc_auc':float(roc_auc_score(y,p)) if len(np.unique(y))==2 else None,
      'average_precision':float(average_precision_score(y,p)) if y.sum() else None,
      'brier_score':float(brier_score_loss(y,p)),
      'confusion':confusion_matrix(y,pred,labels=[0,1]).tolist()}

def choose_threshold(y,p):
    scores=np.array([f1_score(y,np.asarray(p)>=t,average='macro',zero_division=0)
                     for t in CONFIG['threshold_grid']])
    best=np.flatnonzero(np.isclose(scores,scores.max(),atol=1e-12,rtol=0))
    chosen=min(best,key=lambda i:(abs(CONFIG['threshold_grid'][i]-.5),CONFIG['threshold_grid'][i]))
    return CONFIG['threshold_grid'][chosen]

def sigmoid_np(x):return 1/(1+np.exp(-np.clip(x,-40,40)))

class UnionFind:
    def __init__(self,n):self.parent=list(range(n));self.rank=[0]*n
    def find(self,a):
        while self.parent[a]!=a:self.parent[a]=self.parent[self.parent[a]];a=self.parent[a]
        return a
    def union(self,a,b):
        a,b=self.find(a),self.find(b)
        if a==b:return
        if self.rank[a]<self.rank[b]:a,b=b,a
        self.parent[b]=a
        if self.rank[a]==self.rank[b]:self.rank[a]+=1

def stable_groups(keys):
    uf=UnionFind(len(keys));seen={}
    for i,row in enumerate(keys):
        for key in row:
            if key is None:continue
            if key in seen:uf.union(i,seen[key])
            else:seen[key]=i
    members=defaultdict(list)
    for i in range(len(keys)):members[uf.find(i)].append(i)
    ids={root:hashlib.sha256('|'.join(map(str,ix)).encode()).hexdigest()[:20]
         for root,ix in members.items()}
    return np.array([ids[uf.find(i)] for i in range(len(keys))])

def choose_group_split(frame,index,fraction,seed,require_platforms=False,candidates=240):
    index=np.asarray(index,int);y=frame.label_final.to_numpy(int);g=frame.group_id.to_numpy(str)
    available=set(frame.iloc[index].platform);target=y[index].mean();best=None
    # Kandidat dipilih hanya dari jumlah/label/platform, sebelum pelatihan atau prediksi.
    base_props=frame.iloc[index].platform.value_counts(normalize=True)
    for frac in sorted(set([fraction,max(.05,fraction-.04),min(.45,fraction+.04)])):
        splitter=GroupShuffleSplit(n_splits=candidates//3,test_size=frac,random_state=seed)
        for k,(a,b) in enumerate(splitter.split(index,y[index],g[index])):
            fit,hold=index[a],index[b]
            if len(np.unique(y[fit]))<2 or len(np.unique(y[hold]))<2:continue
            if min(np.bincount(y[hold],minlength=2))<3:continue
            if require_platforms and (set(frame.iloc[fit].platform)!=available or set(frame.iloc[hold].platform)!=available):continue
            props=frame.iloc[hold].platform.value_counts(normalize=True).reindex(base_props.index,fill_value=0)
            penalty=4*abs(len(hold)/len(index)-fraction)+abs(y[hold].mean()-target)+float(abs(props-base_props).mean())
            item=(penalty,fit,hold,{'seed':seed,'fraction':frac,'candidate':k})
            if best is None or item[0]<best[0]:best=item
    if best is None:raise RuntimeError('Tidak ditemukan split kelompok yang memenuhi kriteria. Jangan mengganti ke random row split.')
    assert not (set(g[best[1]])&set(g[best[2]]))
    return best[1],best[2],best[3]

def group_weights(groups):
    counts=Counter(groups);w=np.array([1/counts[g] for g in groups],float)
    return w*(len(w)/w.sum())

def fit_lr(X,y,groups=None,C=.2):
    model=make_pipeline(StandardScaler(),LogisticRegression(C=C,max_iter=2000,random_state=42))
    kwargs={} if groups is None else {'logisticregression__sample_weight':group_weights(groups)}
    model.fit(X,y,**kwargs);return model



## 3 Audit label dan perbaikan split teks

ID video YouTube pada query URL dipertahankan. Author diberi scope platform. Semua platform harus ada pada train, validation, dan test. Model dievaluasi pada subset high-agreement yang dinyatakan eksplisit. Angka kappa tidak membuktikan proses anotasi manusia; tidak ada label yang diperbarui.


In [ ]:
import emoji

EMOJI_MAP={'🤡':' badut ','🗿':' batu ','🤮':' muntah ','💩':' kotoran ','💀':' tengkorak ',
 '👎':' jempol bawah ','🖕':' jari tengah ','🤬':' mengumpat ','😡':' marah ','😭':' menangis ',
 '🤣':' tertawa ','😂':' tertawa ','😹':' tertawa ','🔥':' api ','🙏':' terima kasih '}
SLANG={'yg':'yang','dgn':'dengan','bgt':'banget','ga':'tidak','gak':'tidak','gk':'tidak',
 'klo':'kalau','kalo':'kalau','tp':'tapi','tpi':'tapi','udh':'sudah','udah':'sudah',
 'blm':'belum','blom':'belum','bkn':'bukan','krn':'karena','emg':'memang','jg':'juga','kls':'kelas'}

def preprocess_text(value):
    if not present(value):return ''
    text=html.unescape(str(value))
    for key,val in EMOJI_MAP.items():text=text.replace(key,val)
    text=emoji.demojize(text,language='en')
    text=re.sub(r':([a-zA-Z0-9_]+):',lambda m:' '+m.group(1).replace('_',' ')+' ',text)
    text=re.sub(r'https?://\S+|www\.\S+',' HTTPURL ',text)
    text=re.sub(r'@\w+',' @USER ',text)
    text=re.sub(r'#(?=\w)','',text).lower()
    text=re.sub(r'(.)\1{2,}',r'\1\1',text)
    for key,val in SLANG.items():text=re.sub(r'\b'+re.escape(key)+r'\b',val,text)
    return re.sub(r'\s+',' ',text).strip()

def canonical_source_url(value):
    if not present(value):return None
    v=str(value).strip()
    if not re.match(r'^https?://',v):return None
    u=urlsplit(v);host=u.netloc.lower().removeprefix('www.').removeprefix('m.')
    path=re.sub(r'/+','/',unquote(u.path)).rstrip('/')
    query=parse_qs(u.query)
    if host in ('youtube.com','music.youtube.com'):
        vid=query.get('v',[None])[0]
        if not vid:
            match=re.match(r'^/(?:shorts|embed|live)/([^/]+)',path)
            vid=match.group(1) if match else None
        return 'youtube:'+vid if vid else None
    if host=='youtu.be':return 'youtube:'+path.strip('/').split('/')[0] if path.strip('/') else None
    if host in ('twitter.com','x.com'):
        match=re.search(r'/status/(\d+)',path)
        return 'x:status:'+match.group(1) if match else None
    if host.endswith('facebook.com'):
        if path.lower() in ('','/public_group','/posts','/profile'):return None
        meaningful={k:query[k][0] for k in ('story_fbid','fbid','id','v') if k in query}
        if path.lower() in ('/watch','/permalink.php','/photo.php','/story.php','/profile.php'):
            if not meaningful:return None
            return 'facebook:'+path+'?'+urlencode(sorted(meaningful.items()))
        if '/posts/' in path or '/videos/' in path or '/reel/' in path or '/permalink/' in path:
            return 'facebook:'+path
        return None
    if host.endswith('instagram.com'):
        match=re.match(r'^/(p|reel|reels|tv)/([^/]+)',path)
        return 'instagram:'+match.group(2) if match else None
    if host.endswith('tiktok.com'):
        match=re.search(r'/video/(\d+)',path)
        return 'tiktok:'+match.group(1) if match else None
    if not host or path.lower() in ('','/','/posts','/profile','/public_group'):return None
    tracking={'utm_source','utm_medium','utm_campaign','utm_term','utm_content','fbclid','gclid','ref','s','t'}
    keep={k:v[0] for k,v in query.items() if k.lower() not in tracking}
    return host+path+('?' + urlencode(sorted(keep.items())) if keep else '')

def author_key(platform,value):
    if not present(value):return None
    value=str(value).strip().casefold()
    if value in ('anonymous','unknown','fb_user_real','user','pengguna','anon','n/a'):return None
    return 'author:'+str(platform).casefold()+':'+value

MASTER_PATH=PROJECT_ROOT/'data/dataset_master_13307_labeled.csv'
MASTER=pd.read_csv(MASTER_PATH,dtype={'master_id':str},low_memory=False)
assert len(MASTER)==13307 and MASTER.master_id.is_unique
ANN=[f'anotator_{i}' for i in range(1,6)]
assert MASTER[ANN].isin([0,1]).all().all()
assert np.array_equal((MASTER[ANN].sum(axis=1)>=3).astype(int),MASTER.label_final.to_numpy(int))
master_hash=sha256(MASTER_PATH)

# Configuration is locked before training. Re-running the same notebook resumes this directory.
lock={'config':CONFIG,'master_sha256':master_hash}
lock_path=RUN/'LOCKED_CONFIG.json'
if lock_path.exists():
    assert json.loads(lock_path.read_text())==lock,'Konfigurasi/data berbeda. Gunakan run_directory baru; hasil lama dilindungi.'
else:atomic_json(lock,lock_path)
versions={name:importlib.metadata.version(name) for name in ['torch','torchvision','transformers','scikit-learn','pandas','numpy','einops','emoji']}
atomic_json({'versions':versions,'python':sys.version,'gpu':GPU_NAME,'amp':str(AMP_DTYPE),
 'cuda':torch.version.cuda,'started_utc':pd.Timestamp.now(tz='UTC').isoformat()},RUN/'ENVIRONMENT.json')

votes=MASTER[ANN].sum(axis=1).to_numpy();counts=np.column_stack([5-votes,votes]);pi=((counts*counts).sum(1)-5)/20
pj=counts.sum(axis=0)/(len(MASTER)*5);fleiss=float((pi.mean()-(pj*pj).sum())/(1-(pj*pj).sum()))
expert=MASTER.dropna(subset=['pakar_1','pakar_2'])
label_audit={'master_n':len(MASTER),'master_sha256':master_hash,'label_counts':MASTER.label_final.value_counts().to_dict(),
 'fleiss_kappa':fleiss,'majority_consistency':True,'expert_n':len(expert),
 'expert_kappa_current_columns':float(cohen_kappa_score(expert.pakar_1,expert.pakar_2)),
 'human_provenance_status':'REQUIRES_DOCUMENTARY_REVIEW; numeric consistency is not proof of independent human annotation',
 'label_edits_performed':False}
atomic_json(label_audit,RUN/'LABEL_AUDIT.json')

TEXT=MASTER[(MASTER.modality_type=='text_corpus') & (MASTER.derajat_kesepakatan!='Majority (3/5)')].copy()
text_original_n=len(TEXT)
TEXT['text_clean']=TEXT.raw_text_or_ocr.map(preprocess_text)
TEXT=TEXT[TEXT.text_clean.str.len()>0].sort_values('master_id',kind='stable')
conflicts=TEXT.groupby('text_clean').label_final.nunique();ambiguous=set(conflicts[conflicts>1].index)
conflict_n=int(TEXT.text_clean.isin(ambiguous).sum())
TEXT=TEXT[~TEXT.text_clean.isin(ambiguous)].drop_duplicates('text_clean',keep='first').reset_index(drop=True)
TEXT['text_sha256']=TEXT.text_clean.map(lambda s:hashlib.sha256(s.encode()).hexdigest())
keys=[[canonical_source_url(r.source_url),author_key(r.platform,r.author_or_uploader),'text:'+r.text_sha256]
      for r in TEXT.itertuples(index=False)]
TEXT['group_id']=stable_groups(keys)
trainval,test_idx,text_test_rule=choose_group_split(TEXT,np.arange(len(TEXT)),.15,42,require_platforms=True)
train_idx,val_idx,text_val_rule=choose_group_split(TEXT,trainval,.15/.85,142,require_platforms=True)
TEXT['split']=''
for ids,name in [(train_idx,'train'),(val_idx,'validation'),(test_idx,'test')]:TEXT.loc[ids,'split']=name
assert (TEXT.split!='').all()
assert TEXT.groupby('group_id').split.nunique().max()==1
assert TEXT.groupby('text_sha256').split.nunique().max()==1
for name in ['train','validation','test']:
    assert set(TEXT.loc[TEXT.split==name,'platform'])==set(TEXT.platform)
text_manifest=TEXT[['master_id','platform','label_final','group_id','text_sha256','split']]
text_manifest_path=RUN/'TEXT_SPLIT_MANIFEST.csv'
if text_manifest_path.exists():pd.testing.assert_frame_equal(pd.read_csv(text_manifest_path,dtype={'master_id':str}),text_manifest,check_dtype=False)
else:atomic_csv(text_manifest,text_manifest_path)
atomic_json({'source_text_rows':8906,'high_agreement_before_dedup':text_original_n,
 'n_after_preprocessing_dedup':len(TEXT),'conflicting_rows_removed':conflict_n,
 'test_rule':text_test_rule,'validation_rule':text_val_rule,
 'all_five_platforms_in_all_roles':True,'group_overlap':0,'text_hash_overlap':0,
 'platform_by_split':pd.crosstab(TEXT.platform,TEXT.split).to_dict(),
 'scope':'high-agreement subset; not all master rows; split is internal, corpus previously explored'},RUN/'TEXT_PREFLIGHT.json')
print('Text splits with all platforms:')
display(pd.crosstab(TEXT.platform,TEXT.split))
print('Fleiss kappa:',round(fleiss,6),'| Annotation provenance still requires documentary evidence.')



## 4 Audit pasangan modalitas dan fold video

Cocokkan segment_id, keyframe, OCR dan STT sebelum pelatihan. Pada snapshot yang diperiksa, 450 dari 4.401 baris tidak konsisten; sekitar 3.951 pasangan masuk populasi utama. Kode menghitung ulang dari berkas aktual dan mengekspor alasan eksklusi. Eksklusi tidak menggunakan hasil model. Hash gambar, kelompok sumber, dan setiap peran pelatihan/pengujian diaudit.


In [ ]:
CANONICAL=PROJECT_ROOT/CONFIG['canonical_run_relative']
if not (CANONICAL/'CANONICAL_SPLITS.csv').is_file():
    alternative=PROJECT_ROOT/'V6_FINAL_GPU_RUNS/canonical_efficientnet_b0_v2'
    if (alternative/'CANONICAL_SPLITS.csv').is_file():CANONICAL=alternative
REQUIRED=[CANONICAL/'CANONICAL_SPLITS.csv',CANONICAL/'LOCKED_CONFIG.json',
 PROJECT_ROOT/'features/multimodal_video_clean_metadata.csv',
 PROJECT_ROOT/'features/audio_acoustic_features.npy',
 PROJECT_ROOT/'data/multimodal/segmentation_10s_stride3_v2/multimodal_segments.csv']
for p in REQUIRED:assert p.is_file(),f'Berkas wajib tidak ditemukan: {p}'
CANON=pd.read_csv(CANONICAL/'CANONICAL_SPLITS.csv',dtype={'master_id':str,'source_group':str,'video_id':str})
META=pd.read_csv(PROJECT_ROOT/'features/multimodal_video_clean_metadata.csv',dtype={'master_id':str,'video_id':str,'segment_id':str})
SEG=pd.read_csv(REQUIRED[-1],dtype={'master_id':str,'video_id':str,'segment_id':str})
AUDIO_ALL=np.load(PROJECT_ROOT/'features/audio_acoustic_features.npy',allow_pickle=False)
assert len(CANON)==len(META)==len(SEG)==4401 and AUDIO_ALL.shape==(4401,16)
assert np.isfinite(AUDIO_ALL).all()
assert CANON.master_id.is_unique and META.master_id.is_unique and SEG.master_id.is_unique
assert CANON.master_id.tolist()==META.master_id.tolist(),'Audio/metadata ordering mismatch'
assert SEG.set_index('master_id').loc[META.master_id,'video_id'].tolist()==META.video_id.tolist()
assert np.array_equal(CANON.y.to_numpy(int),MASTER.set_index('master_id').loc[CANON.master_id,'label_final'].to_numpy(int))
VIDEO=CANON.merge(SEG[['master_id','segment_id','ocr_text','stt_transcript','video_id']].rename(columns={'video_id':'segment_video_id'}),
                 on='master_id',validate='one_to_one',how='left')
VIDEO=VIDEO.merge(MASTER[['master_id','raw_text_or_ocr','stt_audio_transcript','visual_keyframe_path']],on='master_id',validate='one_to_one')
assert VIDEO.master_id.tolist()==CANON.master_id.tolist()
assert (VIDEO.video_id==VIDEO.segment_video_id).all()
def normalize_extraction(value):return re.sub(r'\s+',' ',str(value)).strip() if present(value) else ''
frame_ok=VIDEO.frame_name.map(lambda v:Path(str(v)).stem)==VIDEO.segment_id
ocr_ok=(VIDEO.ocr_text.map(normalize_extraction)==VIDEO.raw_text_or_ocr.map(normalize_extraction))
stt_ok=(VIDEO.stt_transcript.map(normalize_extraction)==VIDEO.stt_audio_transcript.map(normalize_extraction))
eligible=frame_ok & ocr_ok & stt_ok
alignment=VIDEO[['master_id','segment_id','platform','source_group','outer_fold','y']].copy()
alignment['frame_identity_match']=frame_ok;alignment['ocr_content_match']=ocr_ok;alignment['stt_content_match']=stt_ok
alignment['eligible_primary']=eligible
atomic_csv(alignment,RUN/'VIDEO_ALIGNMENT_AUDIT.csv')
atomic_csv(alignment.loc[~eligible],RUN/'VIDEO_EXCLUDED_ALIGNMENT.csv')
if eligible.sum()<3000:raise RuntimeError('Terlalu banyak pasangan input tidak cocok. Periksa VIDEO_ALIGNMENT_AUDIT sebelum menggunakan GPU.')
VIDEO=VIDEO.loc[eligible].reset_index(drop=True)
AUDIO=AUDIO_ALL[eligible.to_numpy()].astype(np.float32)
VIDEO['label_final']=VIDEO.y.astype(int);VIDEO['group_id']=VIDEO.source_group.astype(str)
VIDEO['ocr_ok']=VIDEO.ocr_text.map(present).astype(int)
VIDEO['stt_ok']=VIDEO.stt_transcript.map(present).astype(int)
VIDEO['ocr_length']=VIDEO.ocr_text.map(lambda x:len(str(x)) if present(x) else 0).clip(upper=512)
VIDEO['stt_length']=VIDEO.stt_transcript.map(lambda x:len(str(x)) if present(x) else 0).clip(upper=512)
VIDEO['text_present']=(VIDEO.ocr_ok|VIDEO.stt_ok).astype(int)
VIDEO['text_clean']=[preprocess_text(' '.join(str(v) for v in (o,s) if present(v)))
                     for o,s in zip(VIDEO.ocr_text,VIDEO.stt_transcript)]
VIDEO.loc[VIDEO.text_present==0,'text_clean']=''
for column in ['group_id','video_id','image_sha256','frame_name']:
    assert VIDEO.groupby(column).outer_fold.nunique().max()==1,f'Cross-fold leakage: {column}'
assert set(VIDEO.outer_fold)=={0,1,2,3}

FRAME_DIR=LOCAL/'frames';FRAME_DIR.mkdir(exist_ok=True)
archive=CANONICAL/'FRAME_CACHE_3996.zip'
expected=set(VIDEO.frame_name)
if archive.is_file():
    with zipfile.ZipFile(archive) as z:
        assert expected.issubset(set(z.namelist())),'Archive lacks required paired keyframes'
        for name in tqdm(sorted(expected),desc='Staging verified keyframes'):
            assert Path(name).name==name,'Unsafe archive entry'
            dest=FRAME_DIR/name
            if not dest.is_file() or dest.stat().st_size!=z.getinfo(name).file_size:
                temp=dest.with_name(dest.name+'.partial')
                with z.open(name) as src,temp.open('wb') as dst:shutil.copyfileobj(src,dst)
                temp.replace(dest)
else:
    source_frames=PROJECT_ROOT/'data/multimodal/segmentation_10s_stride3_v2/frames'
    for name in tqdm(sorted(expected),desc='Staging frames from Drive'):
        src=source_frames/name;assert src.is_file(),f'Missing keyframe: {name}'
        if not (FRAME_DIR/name).is_file():shutil.copy2(src,FRAME_DIR/name)
for name,h in tqdm(VIDEO[['frame_name','image_sha256']].drop_duplicates().itertuples(index=False,name=None),total=len(expected),desc='Verifying image hashes'):
    assert sha256(FRAME_DIR/name)==h,f'Keyframe hash changed: {name}'
    with Image.open(FRAME_DIR/name) as im:im.verify()

VIDEO_ROLES={};role_rows=[];rules={}
for fold in range(4):
    test=np.flatnonzero(VIDEO.outer_fold.to_numpy()==fold)
    outer=np.flatnonzero(VIDEO.outer_fold.to_numpy()!=fold)
    pool,cal,cal_rule=choose_group_split(VIDEO,outer,.15,500+fold)
    base,meta,meta_rule=choose_group_split(VIDEO,pool,.15/.85,600+fold)
    fit,val,val_rule=choose_group_split(VIDEO,base,.10/.70,700+fold)
    roles={'fit':fit,'early_stop':val,'meta_fit':meta,'calibration':cal,'test':test}
    for a,ia in roles.items():
        for b,ib in roles.items():
            if a>=b:continue
            assert not(set(VIDEO.iloc[ia].group_id)&set(VIDEO.iloc[ib].group_id))
            assert not(set(VIDEO.iloc[ia].image_sha256)&set(VIDEO.iloc[ib].image_sha256))
        for i in ia:role_rows.append({'master_id':VIDEO.iloc[i].master_id,'outer_fold':fold,'role':a,'group_id':VIDEO.iloc[i].group_id})
    assert sum(map(len,roles.values()))==len(VIDEO)
    VIDEO_ROLES[fold]=roles;rules[fold]={'calibration':cal_rule,'meta_fit':meta_rule,'early_stop':val_rule}
    print('Video fold',fold,{k:len(v) for k,v in roles.items()})
atomic_csv(pd.DataFrame(role_rows),RUN/'VIDEO_ROLE_MANIFEST.csv')
atomic_csv(VIDEO[['master_id','segment_id','platform','label_final','group_id','outer_fold','frame_name','image_sha256','ocr_ok','stt_ok','ocr_length','stt_length']],RUN/'VIDEO_PRIMARY_MANIFEST.csv')
input_hashes={str(p.relative_to(PROJECT_ROOT)):sha256(p) for p in REQUIRED}
video_input_lock={'input_hashes':input_hashes,'eligible_ids_sha256':hashlib.sha256('|'.join(VIDEO.master_id).encode()).hexdigest()}
video_lock_path=RUN/'VIDEO_INPUT_LOCK.json'
if video_lock_path.exists():assert json.loads(video_lock_path.read_text())==video_input_lock,'Video inputs changed; choose a new run_directory.'
else:atomic_json(video_input_lock,video_lock_path)
atomic_json({'input_hashes':input_hashes,'source_n':4401,'eligible_primary_n':len(VIDEO),
 'excluded_alignment_n':int((~eligible).sum()),'source_groups':VIDEO.group_id.nunique(),
 'source_videos':VIDEO.video_id.nunique(),'unique_images':VIDEO.image_sha256.nunique(),
 'eligibility_rule':'frame segment identity AND normalized OCR match AND normalized STT match; independent of model outcomes',
 'coverage':[{ 'ocr':int(o),'stt':int(s),'n':int(n)} for (o,s),n in VIDEO.groupby(['ocr_ok','stt_ok']).size().items()],
 'role_rules':rules,'zero_crossfold_group_and_image_overlap':True,
 'evaluation_status':'internal repeated grouped evaluation on a previously explored corpus; not external confirmation'},RUN/'VIDEO_PREFLIGHT.json')
print('Video primary:',len(VIDEO),'| Excluded identity/content mismatches:',int((~eligible).sum()))
display(VIDEO.groupby(['ocr_ok','stt_ok']).size().rename('segments').reset_index())



## 5 Pelatihan encoder dan checkpoint

Encoder teks dan visual dilatih ulang per fold/seed dari pretrained generik. Early stopping memilih epoch pada peran khusus. Fitting meta dan kalibrasi ambang memakai peran berbeda. Checkpoint per epoch memungkinkan resume; jika GPU terputus di tengah epoch, epoch tersebut diulang.


In [ ]:
from torchvision.models import efficientnet_b0,EfficientNet_B0_Weights
from torchvision import transforms as T

class TokenDataset(Dataset):
    def __init__(self,texts,labels,tokenizer,max_length):
        self.enc=tokenizer(list(texts),max_length=max_length,truncation=True,padding='max_length',return_tensors='pt')
        self.labels=torch.tensor(np.asarray(labels,int),dtype=torch.long)
    def __len__(self):return len(self.labels)
    def __getitem__(self,i):return {**{k:v[i] for k,v in self.enc.items()},'labels':self.labels[i]}

def data_loader(ds,batch,shuffle=False,seed=42):
    generator=torch.Generator().manual_seed(seed)
    return DataLoader(ds,batch_size=batch,shuffle=shuffle,num_workers=0,pin_memory=True,generator=generator)

def training_loss(logits,y):return nn.functional.cross_entropy(logits,y,label_smoothing=.05)

def rng_state():
    return {'python':random.getstate(),'numpy':np.random.get_state(),'torch':torch.get_rng_state(),'cuda':torch.cuda.get_rng_state_all()}

def restore_rng(state):
    random.setstate(state['python']);np.random.set_state(state['numpy']);torch.set_rng_state(state['torch']);torch.cuda.set_rng_state_all(state['cuda'])

@torch.inference_mode()
def predict_text_model(model,tokenizer,texts,max_length,batch=32,return_features=True):
    ds=TokenDataset(texts,np.zeros(len(texts),int),tokenizer,max_length)
    ps=[];features=[];model.eval()
    for data in data_loader(ds,batch):
        data={k:v.to(DEVICE,non_blocking=True) for k,v in data.items() if k!='labels'}
        with torch.autocast('cuda',dtype=AMP_DTYPE):
            out=model(**data,output_hidden_states=return_features,return_dict=True)
        ps.append(out.logits.float().softmax(-1)[:,1].cpu().numpy())
        if return_features:features.append(out.hidden_states[-1][:,0,:].float().cpu().numpy())
    return np.concatenate(ps),np.concatenate(features) if features else None

def train_transformer(model_id,frame,fit_index,val_index,pred_index,seed,folder,max_length,epochs):
    folder=Path(folder);folder.mkdir(parents=True,exist_ok=True)
    result_path=folder/'outputs.npz'
    if result_path.is_file():
        with np.load(result_path,allow_pickle=False) as z:
            assert np.array_equal(z['ids'],frame.iloc[pred_index].master_id.to_numpy(str))
            return {k:z[k] for k in z.files}
    seed_all(seed)
    revision=model_revision(model_id)
    tokenizer=AutoTokenizer.from_pretrained(model_id,revision=revision)
    model=AutoModelForSequenceClassification.from_pretrained(model_id,revision=revision,num_labels=2,attn_implementation='sdpa').to(DEVICE)
    tokenized=TokenDataset(frame.iloc[fit_index].text_clean,frame.iloc[fit_index].label_final,tokenizer,max_length)
    steps=math.ceil(math.ceil(len(tokenized)/CONFIG['text_batch'])/CONFIG['accumulation'])*epochs
    optimizer=torch.optim.AdamW(model.parameters(),lr=3e-5,weight_decay=.01)
    scheduler=get_cosine_schedule_with_warmup(optimizer,max(1,int(.1*steps)),steps)
    scaler=torch.amp.GradScaler('cuda',enabled=not USE_BF16)
    checkpoint=folder/'resume.pt';best_path=folder/'best_model.pt'
    start=0;best=-float('inf');history=[]
    if checkpoint.is_file():
        # This is our own checkpoint created in the locked run directory.
        cp=torch.load(checkpoint,map_location='cpu',weights_only=False)
        model.load_state_dict(cp['model']);optimizer.load_state_dict(cp['optimizer']);scheduler.load_state_dict(cp['scheduler'])
        scaler.load_state_dict(cp['scaler']);start=cp['epoch'];best=cp['best'];history=cp['history'];restore_rng(cp['rng'])
        for state in optimizer.state.values():
            for k,v in state.items():
                if isinstance(v,torch.Tensor):state[k]=v.to(DEVICE)
        del cp
        print('Resume transformer:',folder.name,'epoch',start)
    for epoch in range(start,epochs):
        loader=data_loader(tokenized,CONFIG['text_batch'],shuffle=True,seed=seed+epoch)
        model.train();optimizer.zero_grad(set_to_none=True);total=0
        for step,batch in enumerate(tqdm(loader,desc=f'{folder.name} epoch {epoch+1}/{epochs}',leave=False)):
            y=batch.pop('labels').to(DEVICE);batch={k:v.to(DEVICE,non_blocking=True) for k,v in batch.items()}
            with torch.autocast('cuda',dtype=AMP_DTYPE):
                logits=model(**batch).logits
                window_start=(step//CONFIG['accumulation'])*CONFIG['accumulation']
                window_length=min(CONFIG['accumulation'],len(loader)-window_start)
                loss=training_loss(logits,y)/window_length
            scaler.scale(loss).backward();total+=float(loss.item())*window_length
            if (step+1)%CONFIG['accumulation']==0 or step+1==len(loader):
                scaler.unscale_(optimizer);nn.utils.clip_grad_norm_(model.parameters(),1.0)
                scaler.step(optimizer);scaler.update();scheduler.step();optimizer.zero_grad(set_to_none=True)
        pv,_=predict_text_model(model,tokenizer,frame.iloc[val_index].text_clean,max_length,return_features=False)
        score=f1_score(frame.iloc[val_index].label_final,pv>=.5,average='macro',zero_division=0)
        improved=score>best+1e-12
        if improved:best=float(score);atomic_torch(cpu_state(model),best_path)
        history.append({'epoch':epoch+1,'fit_loss':total/max(len(loader),1),'early_stop_macro_f1_at_050':float(score)})
        atomic_json(history,folder/'TRAINING_HISTORY.json')
        atomic_torch({'model':cpu_state(model),'optimizer':optimizer.state_dict(),'scheduler':scheduler.state_dict(),
            'scaler':scaler.state_dict(),'epoch':epoch+1,'best':best,'history':history,'rng':rng_state()},checkpoint)
        print(folder.name,'epoch',epoch+1,'validation F1',round(float(score),4),flush=True)
    model.load_state_dict(torch.load(best_path,map_location='cpu',weights_only=True))
    p,embedding=predict_text_model(model,tokenizer,frame.iloc[pred_index].text_clean,max_length)
    result={'ids':frame.iloc[pred_index].master_id.to_numpy(str),'probability':p,'embedding':embedding.astype(np.float32)}
    atomic_npz(result_path,**result)
    atomic_json({'model_id':model_id,'resolved_commit':getattr(model.config,'_commit_hash',None),'seed':seed,
        'epochs':epochs,'max_length':max_length,'selection':'best early-stop-role macro F1 at fixed 0.5; earliest tie',
        'fit_n':len(fit_index),'early_stop_n':len(val_index)},folder/'MODEL_MANIFEST.json')
    tokenizer.save_pretrained(folder/'tokenizer')
    checkpoint.unlink(missing_ok=True)
    if not CONFIG['save_best_weights']:best_path.unlink(missing_ok=True)
    del model,optimizer,scheduler,tokenized;cleanup();return result

TRAIN_TRANSFORM=T.Compose([T.Resize((256,256)),T.RandomResizedCrop(224,scale=(.85,1.0)),
 T.ColorJitter(brightness=.08,contrast=.08),T.ToTensor(),T.Normalize([.485,.456,.406],[.229,.224,.225])])
EVAL_TRANSFORM=T.Compose([T.Resize((256,256)),T.CenterCrop(224),T.ToTensor(),T.Normalize([.485,.456,.406],[.229,.224,.225])])

class FrameDataset(Dataset):
    def __init__(self,index,transform):self.index=np.asarray(index,int);self.transform=transform
    def __len__(self):return len(self.index)
    def __getitem__(self,i):
        row=VIDEO.iloc[self.index[i]]
        with Image.open(FRAME_DIR/row.frame_name) as im:x=self.transform(im.convert('RGB'))
        return x,int(row.label_final)

@torch.inference_mode()
def predict_visual(model,index):
    model.eval();ps=[];emb=[]
    for x,y in data_loader(FrameDataset(index,EVAL_TRANSFORM),64):
        with torch.autocast('cuda',dtype=AMP_DTYPE):
            features=model.avgpool(model.features(x.to(DEVICE))).flatten(1)
            logits=model.classifier(features)
        ps.append(logits.float().softmax(-1)[:,1].cpu().numpy());emb.append(features.float().cpu().numpy())
    return np.concatenate(ps),np.concatenate(emb)

def train_visual(fit_index,val_index,seed,folder):
    folder=Path(folder);folder.mkdir(parents=True,exist_ok=True);out=folder/'outputs.npz'
    if out.is_file():
        with np.load(out,allow_pickle=False) as z:
            assert np.array_equal(z['ids'],VIDEO.master_id.to_numpy(str))
            return {k:z[k] for k in z.files}
    seed_all(seed);model=efficientnet_b0(weights=EfficientNet_B0_Weights.DEFAULT)
    for p in model.parameters():p.requires_grad=False
    for k in (7,8):
        for p in model.features[k].parameters():p.requires_grad=True
    model.classifier=nn.Sequential(nn.Dropout(.2),nn.Linear(model.classifier[1].in_features,2));model.to(DEVICE)
    optimizer=torch.optim.AdamW([{'params':list(model.features[7].parameters())+list(model.features[8].parameters()),'lr':2e-5},
        {'params':model.classifier.parameters(),'lr':3e-4}],weight_decay=.01)
    scaler=torch.amp.GradScaler('cuda',enabled=not USE_BF16);start=0;best=-float('inf');history=[]
    cp_path=folder/'resume.pt';best_path=folder/'best_model.pt'
    if cp_path.is_file():
        cp=torch.load(cp_path,map_location='cpu',weights_only=False);model.load_state_dict(cp['model']);optimizer.load_state_dict(cp['optimizer'])
        scaler.load_state_dict(cp['scaler']);start=cp['epoch'];best=cp['best'];history=cp['history'];restore_rng(cp['rng'])
        for state in optimizer.state.values():
            for k,v in state.items():
                if isinstance(v,torch.Tensor):state[k]=v.to(DEVICE)
        del cp
    for epoch in range(start,CONFIG['visual_epochs']):
        loader=data_loader(FrameDataset(fit_index,TRAIN_TRANSFORM),CONFIG['visual_batch'],True,seed+epoch)
        model.train()
        for layer in model.modules():
            if isinstance(layer,nn.modules.batchnorm._BatchNorm):layer.eval()
        total=0
        for x,y in tqdm(loader,desc=f'{folder.name} visual {epoch+1}',leave=False):
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast('cuda',dtype=AMP_DTYPE):loss=training_loss(model(x.to(DEVICE)),y.to(DEVICE))
            scaler.scale(loss).backward();scaler.unscale_(optimizer);nn.utils.clip_grad_norm_(model.parameters(),1.0)
            scaler.step(optimizer);scaler.update();total+=float(loss)
        pv,_=predict_visual(model,val_index);score=f1_score(VIDEO.iloc[val_index].label_final,pv>=.5,average='macro',zero_division=0)
        if score>best+1e-12:best=float(score);atomic_torch(cpu_state(model),best_path)
        history.append({'epoch':epoch+1,'fit_loss':total/max(len(loader),1),'early_stop_macro_f1_at_050':float(score)})
        atomic_json(history,folder/'TRAINING_HISTORY.json')
        atomic_torch({'model':cpu_state(model),'optimizer':optimizer.state_dict(),'scaler':scaler.state_dict(),
         'epoch':epoch+1,'best':best,'history':history,'rng':rng_state()},cp_path)
        print(folder.name,'visual epoch',epoch+1,'validation F1',round(float(score),4),flush=True)
    model.load_state_dict(torch.load(best_path,map_location='cpu',weights_only=True))
    p,e=predict_visual(model,np.arange(len(VIDEO)))
    result={'ids':VIDEO.master_id.to_numpy(str),'probability':p,'embedding':e.astype(np.float32)}
    atomic_npz(out,**result)
    atomic_json({'architecture':'torchvision EfficientNet-B0','trainable_blocks':[7,8],'classifier':[.2,1280,2],
      'seed':seed,'epochs':CONFIG['visual_epochs'],'selection':'early-stop role fixed-0.5 macro F1'},folder/'MODEL_MANIFEST.json')
    cp_path.unlink(missing_ok=True)
    if not CONFIG['save_best_weights']:best_path.unlink(missing_ok=True)
    del model,optimizer;cleanup();return result



## 6 Pembanding 2024 dan 2025 serta fungsi fusi

HEALNet memakai modul resmi yang dikunci commit dan hash, diadaptasi pada fitur tiap fold. SigLIP 2 menggunakan checkpoint resmi Google sebagai frozen representation baseline. Keduanya dilaporkan sesuai bentuk adaptasi; tidak disebut reproduksi hasil asli atau bukti SOTA. Source HEALNet dan lisensinya disimpan bersama paket.


In [ ]:
import urllib.request,importlib.util,joblib

HEALNET_SHA256='78e9c2b10455f6ad43a88ed4e0a646cbf6304bf52cac06a5140f49dbc4796820'
HealNet=None
if CONFIG['run_recent_baselines']:
    vendor=RUN/'vendor';vendor.mkdir(exist_ok=True)
    source=vendor/'healnet_official.py'
    if not source.is_file():
        url=f"https://raw.githubusercontent.com/konst-int-i/healnet/{CONFIG['healnet_commit']}/healnet/models/healnet.py"
        with urllib.request.urlopen(url,timeout=60) as stream:data=stream.read()
        assert hashlib.sha256(data).hexdigest()==HEALNET_SHA256,'Official HEALNet source hash mismatch'
        source.write_bytes(data)
    assert sha256(source)==HEALNET_SHA256
    license_path=vendor/'HEALNET_LICENSE'
    if not license_path.is_file():
        with urllib.request.urlopen(f"https://raw.githubusercontent.com/konst-int-i/healnet/{CONFIG['healnet_commit']}/LICENSE",timeout=60) as stream:license_path.write_bytes(stream.read())
    spec=importlib.util.spec_from_file_location('v6_healnet_official',source)
    official=importlib.util.module_from_spec(spec);spec.loader.exec_module(official);HealNet=official.HealNet
    atomic_json({'source':'https://github.com/konst-int-i/healnet','commit':CONFIG['healnet_commit'],
      'sha256':HEALNET_SHA256,'adaptation':'official fusion module; one vector token per modality; zero vector for absent text; frozen fold-specific encoders',
      'configuration':{'depth':2,'latent_count':16,'latent_width':64,'heads':4,'head_dim':16,'dropout':.1},
      'claim':'local feature-level adaptation; not reproduction of biomedical paper scores or end-to-end baseline'},vendor/'HEALNET_PROVENANCE.json')

def extract_siglip2():
    dest=RUN/'siglip2_frozen_features.npz'
    if dest.is_file():
        with np.load(dest,allow_pickle=False) as z:
            assert np.array_equal(z['ids'],VIDEO.master_id.to_numpy(str))
            return {k:z[k] for k in z.files}
    revision=model_revision(CONFIG['siglip2_model'])
    processor=AutoProcessor.from_pretrained(CONFIG['siglip2_model'],revision=revision)
    model=AutoModel.from_pretrained(CONFIG['siglip2_model'],revision=revision,attn_implementation='sdpa').to(DEVICE).eval()
    visual=[];text=[];batch=32
    with torch.inference_mode():
        for start in tqdm(range(0,len(VIDEO),batch),desc='SigLIP2 frozen image and text features'):
            rows=VIDEO.iloc[start:start+batch];images=[]
            for name in rows.frame_name:
                with Image.open(FRAME_DIR/name) as im:images.append(im.convert('RGB').copy())
            vi=processor(images=images,return_tensors='pt').to(DEVICE)
            tx=processor(text=rows.text_clean.str.lower().tolist(),padding='max_length',truncation=True,max_length=64,return_tensors='pt').to(DEVICE)
            with torch.autocast('cuda',dtype=AMP_DTYPE):
                v=model.get_image_features(**vi);t=model.get_text_features(**tx)
            if not isinstance(v,torch.Tensor) or not isinstance(t,torch.Tensor):raise TypeError('Unexpected SigLIP2 feature API output; use pinned transformers version.')
            visual.append(nn.functional.normalize(v.float(),dim=-1).cpu().numpy())
            text.append(nn.functional.normalize(t.float(),dim=-1).cpu().numpy())
    vf=np.concatenate(visual).astype(np.float32);tf=np.concatenate(text).astype(np.float32)
    tf[VIDEO.text_present.to_numpy()==0]=0
    out={'ids':VIDEO.master_id.to_numpy(str),'visual':vf,'text':tf}
    assert len(vf)==len(tf)==len(VIDEO) and np.isfinite(vf).all() and np.isfinite(tf).all()
    atomic_npz(dest,**out)
    atomic_json({'checkpoint':CONFIG['siglip2_model'],'resolved_commit':getattr(model.config,'_commit_hash',None),
     'source':'https://huggingface.co/google/siglip2-base-patch16-224','training':'frozen pretrained features; supervised linear probe on fit role only',
     'text_max_length':64,'absent_text':'zero feature vector','purpose':'2025 vision-language representation baseline; does not isolate fusion architecture'},RUN/'SIGLIP2_PROVENANCE.json')
    del model,processor;cleanup();return out

class FeatureFusion(nn.Module):
    def __init__(self,dimensions,kind):
        super().__init__();self.kind=kind
        if kind=='healnet':
            self.core=HealNet(n_modalities=3,channel_dims=list(dimensions),num_spatial_axes=[1,1,1],out_dims=2,
              depth=2,l_c=16,l_d=64,x_heads=4,l_heads=4,cross_dim_head=16,latent_dim_head=16,
              attn_dropout=.1,ff_dropout=.1,fourier_encode_data=False,snn=False)
        else:self.core=nn.Sequential(nn.Linear(sum(dimensions),128),nn.ReLU(),nn.Dropout(.2),nn.Linear(128,32),nn.ReLU(),nn.Linear(32,2))
    def forward(self,inputs):
        return self.core([x.unsqueeze(1) for x in inputs]) if self.kind=='healnet' else self.core(torch.cat(inputs,dim=1))

def train_feature_fusion(features,meta_index,seed,folder,kind='concat_mlp'):
    folder=Path(folder);folder.mkdir(parents=True,exist_ok=True);path=folder/'outputs.npz'
    if path.is_file():
        with np.load(path,allow_pickle=False) as z:
            assert np.array_equal(z['ids'],VIDEO.master_id.to_numpy(str));return z['probability']
    seed_all(seed);scalers=[StandardScaler().fit(x[meta_index]) for x in features]
    arrays=[sc.transform(x).astype(np.float32) for sc,x in zip(scalers,features)]
    arrays[2][VIDEO.text_present.to_numpy()==0]=0
    model=FeatureFusion([x.shape[1] for x in arrays],kind).to(DEVICE)
    optimizer=torch.optim.AdamW(model.parameters(),lr=CONFIG['head_lr'],weight_decay=.01)
    labels=VIDEO.label_final.to_numpy(int)
    sample_weight=group_weights(VIDEO.iloc[meta_index].group_id.to_numpy())
    epochs=CONFIG['head_epochs'];loss_history=[]
    cp_path=folder/'resume.pt';start=0
    if cp_path.is_file():
        cp=torch.load(cp_path,map_location='cpu',weights_only=False);model.load_state_dict(cp['model']);optimizer.load_state_dict(cp['optimizer'])
        start=cp['epoch'];loss_history=cp['history'];restore_rng(cp['rng'])
        for state in optimizer.state.values():
            for k,v in state.items():
                if isinstance(v,torch.Tensor):state[k]=v.to(DEVICE)
        del cp
    for epoch in range(start,epochs):
        model.train();order=np.random.default_rng(seed+epoch).permutation(len(meta_index));total=0
        for startpos in range(0,len(order),CONFIG['head_batch']):
            ix=order[startpos:startpos+CONFIG['head_batch']];pos=meta_index[ix]
            inputs=[torch.from_numpy(x[pos]).to(DEVICE) for x in arrays]
            y=torch.from_numpy(labels[pos]).to(DEVICE);w=torch.from_numpy(sample_weight[ix]).float().to(DEVICE)
            optimizer.zero_grad(set_to_none=True);logits=model(inputs)
            assert logits.shape==(len(pos),2) and torch.isfinite(logits).all()
            loss=(nn.functional.cross_entropy(logits,y,reduction='none')*w).mean();loss.backward()
            if kind=='healnet' and epoch==0 and startpos==0:
                # Official source has an internal catch; require effective gradients through every modality.
                for modality in range(3):
                    prefix=f'core.layers.0.{2*modality}.'
                    grads=[p.grad for n,p in model.named_parameters() if n.startswith(prefix) and p.grad is not None]
                    assert grads and any(float(g.abs().sum())>0 for g in grads),f'HEALNet modality {modality} produced no gradient'
            nn.utils.clip_grad_norm_(model.parameters(),1.0);optimizer.step();total+=float(loss)
        loss_history.append({'epoch':epoch+1,'meta_fit_loss':total/max(math.ceil(len(meta_index)/CONFIG['head_batch']),1)})
        atomic_torch({'model':cpu_state(model),'optimizer':optimizer.state_dict(),'epoch':epoch+1,'history':loss_history,'rng':rng_state()},cp_path)
    model.eval();prob=[]
    with torch.inference_mode():
        for startpos in range(0,len(VIDEO),256):
            inputs=[torch.from_numpy(x[startpos:startpos+256]).to(DEVICE) for x in arrays]
            prob.append(model(inputs).float().softmax(-1)[:,1].cpu().numpy())
    p=np.concatenate(prob)
    atomic_npz(path,ids=VIDEO.master_id.to_numpy(str),probability=p)
    atomic_torch(cpu_state(model),folder/'model.pt');joblib.dump(scalers,folder/'scalers.joblib')
    atomic_json({'kind':kind,'feature_dims':[x.shape[1] for x in arrays],'epochs':epochs,
     'training_role':'meta_fit only; fixed epochs; no outer-test or calibration labels used in fitting',
     'seed':seed,'history':loss_history},folder/'MODEL_MANIFEST.json')
    cp_path.unlink(missing_ok=True);del model,optimizer;cleanup();return p

def coverage_features(frame,pv,pa,pt,variant):
    if variant=='va':return np.column_stack([pv,pa])
    base=[pv,pa,pt]
    if variant=='vat':return np.column_stack(base)
    o=frame.ocr_ok.to_numpy(float);s=frame.stt_ok.to_numpy(float)
    ol=np.log1p(frame.ocr_length.to_numpy(float));sl=np.log1p(frame.stt_length.to_numpy(float))
    if variant=='coverage_only':return np.column_stack([o,s,ol,sl])
    base.extend([o,s])
    if variant=='coverage':base.extend([ol,sl,pt*o,pt*s,pv*o,pa*s])
    return np.column_stack(base)

SIGLIP=None
if CONFIG['run_recent_baselines'] or CONFIG['run_lopo_probe']:SIGLIP=extract_siglip2()



## 7 Eksperimen teks pada split yang benar

Pembanding: TF-IDF word+character LR, IndoBERT-p2, IndoBERTweet asli, mBERT, dan equal-vote ensemble. IndoBERT-p2 juga diulang pada dua seed tambahan. Semua model memakai split yang sama dan ambang dipilih pada validation. Hasil test seed terbaik tidak dipilih.


In [ ]:
TEXT_RESULTS=[];TEXT_PREDICTIONS=[]
if CONFIG['run_text']:
    base_folder=RUN/'text';base_folder.mkdir(exist_ok=True)
    text_tasks=[(name,model,seed) for seed in CONFIG['text_seeds'] for name,model in CONFIG['text_models'].items()]
    text_tasks += [('indobert_p2',CONFIG['text_models']['indobert_p2'],s) for s in CONFIG['text_primary_extra_seeds'] if s not in CONFIG['text_seeds']]
    indices={k:np.flatnonzero(TEXT.split.to_numpy()==k) for k in ['train','validation','test']}
    all_index=np.arange(len(TEXT));result_map={}
    for name,model_id,seed in text_tasks:
        folder=base_folder/f'{name}_seed_{seed}'
        out=train_transformer(model_id,TEXT,indices['train'],indices['validation'],all_index,seed,folder,
                              CONFIG['text_max_length'],CONFIG['text_epochs'])
        result_map[(name,seed)]=out['probability']
    # Lexical comparison trained on the same corrected groups.
    lexical_path=base_folder/'tfidf_predictions.npz'
    if lexical_path.is_file():
        with np.load(lexical_path,allow_pickle=False) as z:
            assert np.array_equal(z['ids'],TEXT.master_id.to_numpy(str));lexical_p=z['probability']
    else:
        lexical=Pipeline([('features',FeatureUnion([
           ('word',TfidfVectorizer(ngram_range=(1,2),min_df=2,max_features=40000,sublinear_tf=True)),
           ('char',TfidfVectorizer(analyzer='char',ngram_range=(3,5),min_df=2,max_features=60000,sublinear_tf=True))])),
           ('classifier',LogisticRegression(C=1.0,max_iter=2000,random_state=42))])
        lexical.fit(TEXT.iloc[indices['train']].text_clean,TEXT.iloc[indices['train']].label_final)
        lexical_p=lexical.predict_proba(TEXT.text_clean)[:,1]
        atomic_npz(lexical_path,ids=TEXT.master_id.to_numpy(str),probability=lexical_p)
        joblib.dump(lexical,base_folder/'tfidf_model.joblib')
    result_map[('tfidf_word_char_lr',42)]=lexical_p
    for seed in CONFIG['text_seeds']:
        components=[result_map[(name,seed)] for name in CONFIG['text_models']]
        if len(components)>1:result_map[('equal_text_ensemble',seed)]=np.mean(components,axis=0)
    for (name,seed),p in result_map.items():
        val=indices['validation'];test=indices['test'];threshold=choose_threshold(TEXT.iloc[val].label_final,p[val])
        m=metrics(TEXT.iloc[test].label_final,p[test],threshold=threshold)
        fixed=metrics(TEXT.iloc[test].label_final,p[test],threshold=.5)
        TEXT_RESULTS.append({'model':name,'seed':seed,'threshold':threshold,'threshold_source':'validation only',
            **m,'macro_f1_fixed_050':fixed['macro_f1'],'accuracy_fixed_050':fixed['accuracy']})
        d=TEXT.iloc[test][['master_id','platform','group_id','label_final']].copy()
        d['model']=name;d['seed']=seed;d['probability']=p[test];d['threshold']=threshold;d['prediction']=(p[test]>=threshold).astype(int)
        TEXT_PREDICTIONS.append(d)
    atomic_csv(pd.DataFrame(TEXT_RESULTS),RUN/'TEXT_TEST_METRICS.csv')
    atomic_csv(pd.concat(TEXT_PREDICTIONS,ignore_index=True),RUN/'TEXT_TEST_PREDICTIONS.csv')
    summary=pd.DataFrame(TEXT_RESULTS).groupby('model')[['macro_f1','accuracy','roc_auc']].agg(['mean','std','count'])
    summary.columns=['_'.join(c) for c in summary.columns]
    atomic_csv(summary.reset_index(),RUN/'TEXT_SEED_SUMMARY.csv')
    print('Text comparisons complete; no selection of the best test seed.')
    display(pd.DataFrame(TEXT_RESULTS)[['model','seed','n','threshold','accuracy','macro_f1','roc_auc']])



## 8 Eksperimen multimodal utama dan ablasi

Empat outer fold, tiga seed. Cabang teks video dilatih pada data video dengan preprocessing yang sama saat inference. Konfigurasi meliputi unimodal, equal fusion, LR, flags, coverage, negative controls, MLP, adapted HEALNet dan SigLIP2 probes. Analisis dropout menghilangkan bukti OCR/STT secara terkontrol dengan ambang tetap; bukan test eksternal baru.


In [ ]:
VIDEO_PREDICTIONS=[];VIDEO_METRICS=[];STRESS_ROWS=[];FOLD_THRESHOLD_ROWS=[]
if CONFIG['run_video']:
    for seed in CONFIG['video_seeds']:
        for fold in range(4):
            folder=RUN/'video'/f'seed_{seed}'/f'fold_{fold}'
            folder.mkdir(parents=True,exist_ok=True);final=folder/'TEST_PREDICTIONS.csv'
            if final.is_file():
                d=pd.read_csv(final,dtype={'master_id':str,'group_id':str})
                assert set(d.master_id)==set(VIDEO.iloc[VIDEO_ROLES[fold]['test']].master_id)
                VIDEO_PREDICTIONS.append(d)
                if (folder/'STRESS.csv').is_file():STRESS_ROWS.extend(pd.read_csv(folder/'STRESS.csv').to_dict('records'))
                print('Resume completed seed/fold:',seed,fold,flush=True);continue
            roles=VIDEO_ROLES[fold];fit=roles['fit'];early=roles['early_stop'];meta=roles['meta_fit'];cal=roles['calibration'];test=roles['test']
            # Missing text never enters the supervised text training loss.
            text_fit=fit[VIDEO.iloc[fit].text_present.to_numpy()==1]
            text_early=early[VIDEO.iloc[early].text_present.to_numpy()==1]
            assert VIDEO.iloc[text_fit].label_final.nunique()==2 and VIDEO.iloc[text_early].label_final.nunique()==2
            tx=train_transformer(CONFIG['text_models']['indobert_p2'],VIDEO,text_fit,text_early,np.arange(len(VIDEO)),seed,
                  folder/'text_branch',CONFIG['video_max_length'],CONFIG['video_text_epochs'])
            vi=train_visual(fit,early,seed,folder/'visual_branch')
            pt=tx['probability'].copy();te=tx['embedding'].copy()
            no_text=VIDEO.text_present.to_numpy()==0;pt[no_text]=.5;te[no_text]=0
            pv=vi['probability'];ve=vi['embedding']
            audio_model=fit_lr(AUDIO[fit],VIDEO.iloc[fit].label_final,VIDEO.iloc[fit].group_id.to_numpy(),C=CONFIG['base_audio_C'])
            pa=audio_model.predict_proba(AUDIO)[:,1];joblib.dump(audio_model,folder/'audio_model.joblib')
            probabilities={'majority':np.full(len(VIDEO),float(VIDEO.iloc[fit].label_final.mean())),
             'text':pt,'visual':pv,'audio':pa,'equal_va':(pv+pa)/2,'equal_vat':(pv+pa+pt)/3}
            # Majority is a fixed class predictor, independent of calibration threshold optimization.
            fitted={}
            for name,variant in [('va_lr','va'),('vat_lr','vat'),('vat_flags','flags'),('vat_coverage','coverage'),('coverage_only','coverage_only')]:
                X=coverage_features(VIDEO,pv,pa,pt,variant)
                clf=fit_lr(X[meta],VIDEO.iloc[meta].label_final,VIDEO.iloc[meta].group_id.to_numpy(),C=CONFIG['fusion_C'])
                probabilities[name]=clf.predict_proba(X)[:,1];fitted[name]=(clf,variant)
                joblib.dump(clf,folder/(name+'.joblib'))
            # Negative control: jointly shuffle availability/length within platform and role; no label used.
            shuffled=VIDEO.copy();columns=['ocr_ok','stt_ok','ocr_length','stt_length']
            for rid,(role,ix) in enumerate(roles.items()):
                for pid,platform in enumerate(sorted(VIDEO.iloc[ix].platform.unique())):
                    pos=ix[VIDEO.iloc[ix].platform.to_numpy()==platform]
                    perm=np.random.default_rng(seed+1000*fold+100*rid+pid).permutation(pos)
                    shuffled.loc[pos,columns]=VIDEO.loc[perm,columns].to_numpy()
            Xsh=coverage_features(shuffled,pv,pa,pt,'coverage')
            sh=fit_lr(Xsh[meta],VIDEO.iloc[meta].label_final,VIDEO.iloc[meta].group_id.to_numpy(),C=CONFIG['fusion_C'])
            probabilities['coverage_shuffled_control']=sh.predict_proba(Xsh)[:,1]
            probabilities['concat_mlp']=train_feature_fusion([ve,AUDIO,te],meta,seed,folder/'concat_mlp')
            if CONFIG['run_recent_baselines']:
                probabilities['healnet_adapted_2024']=train_feature_fusion([ve,AUDIO,te],meta,seed,folder/'healnet_adapted_2024','healnet')
                sig_variants={'siglip2_visual_probe':SIGLIP['visual'],
                  'siglip2_vt_probe':np.column_stack([SIGLIP['visual'],SIGLIP['text']]),
                  'siglip2_vta_probe':np.column_stack([SIGLIP['visual'],SIGLIP['text'],AUDIO])}
                for name,X in sig_variants.items():
                    clf=fit_lr(X[fit],VIDEO.iloc[fit].label_final,VIDEO.iloc[fit].group_id.to_numpy(),C=.2)
                    probabilities[name]=clf.predict_proba(X)[:,1];joblib.dump(clf,folder/(name+'.joblib'))
            thresholds={}
            d=VIDEO.iloc[test][['master_id','platform','group_id','outer_fold','label_final','ocr_ok','stt_ok']].copy()
            d['seed']=seed
            for name,p in probabilities.items():
                assert np.isfinite(p).all() and len(p)==len(VIDEO)
                threshold=.5 if name=='majority' else choose_threshold(VIDEO.iloc[cal].label_final,p[cal])
                thresholds[name]=threshold
                d[name+'_probability']=p[test];d[name+'_threshold']=threshold;d[name+'_prediction']=(p[test]>=threshold).astype(int)
            atomic_json({'seed':seed,'outer_fold':fold,'thresholds':thresholds,
              'threshold_source':'calibration role only; majority fixed at 0.5','roles_n':{k:len(v) for k,v in roles.items()}},folder/'THRESHOLDS.json')
            # Controlled loss of the complete OCR/STT text extraction; original threshold remains fixed.
            if CONFIG['run_missingness_stress']:
                stress=[]
                for fraction in (0.,.25,.50,1.):
                    stressed=VIDEO.copy();stressed_pt=pt.copy()
                    mask=np.array([int(hashlib.sha256(mid.encode()).hexdigest()[:8],16)/16**8<fraction for mid in VIDEO.master_id])
                    stressed.loc[mask,columns]=0;stressed_pt[mask]=.5
                    for name in ['vat_lr','vat_flags','vat_coverage']:
                        clf,variant=fitted[name];ps=clf.predict_proba(coverage_features(stressed,pv,pa,stressed_pt,variant))[:,1]
                        row={'seed':seed,'fold':fold,'model':name,'text_dropout_fraction':fraction,
                             'threshold':thresholds[name],**metrics(VIDEO.iloc[test].label_final,ps[test],threshold=thresholds[name])}
                        stress.append(row)
                atomic_csv(pd.DataFrame(stress),folder/'STRESS.csv');STRESS_ROWS.extend(stress)
            # Commit completed task only after all configurations and diagnostics succeed.
            atomic_csv(d,final);VIDEO_PREDICTIONS.append(d)
            print('Completed seed',seed,'fold',fold,'n_test',len(test),flush=True)
            del tx,vi,ve,te,probabilities,fitted;cleanup()
    VIDEO_PRED=pd.concat(VIDEO_PREDICTIONS,ignore_index=True)
    for seed,df in VIDEO_PRED.groupby('seed'):
        assert len(df)==len(VIDEO) and df.master_id.is_unique
    names=[c[:-12] for c in VIDEO_PRED.columns if c.endswith('_probability')]
    for seed,df in VIDEO_PRED.groupby('seed'):
        for name in names:
            result=metrics(df.label_final,df[name+'_probability'],pred=df[name+'_prediction'])
            fixed=metrics(df.label_final,df[name+'_probability'],threshold=.5)
            VIDEO_METRICS.append({'model':name,'seed':int(seed),**result,'macro_f1_fixed_050':fixed['macro_f1'],
               'accuracy_fixed_050':fixed['accuracy']})
    atomic_csv(VIDEO_PRED,RUN/'VIDEO_OOF_PREDICTIONS.csv')
    atomic_csv(pd.DataFrame(VIDEO_METRICS),RUN/'VIDEO_METRICS_BY_SEED.csv')
    summary=pd.DataFrame(VIDEO_METRICS).groupby('model')[['macro_f1','accuracy','roc_auc','average_precision']].agg(['mean','std'])
    summary.columns=['_'.join(c) for c in summary.columns];atomic_csv(summary.reset_index(),RUN/'VIDEO_SEED_SUMMARY.csv')
    if STRESS_ROWS:atomic_csv(pd.DataFrame(STRESS_ROWS),RUN/'TEXT_EXTRACTION_DROPOUT_STRESS.csv')
    display(pd.DataFrame(VIDEO_METRICS)[['model','seed','n','accuracy','macro_f1','roc_auc','macro_f1_fixed_050']])



## 9 Uncertainty dan pengujian berpasangan

Seed 42 ditetapkan sebagai analisis utama sebelum hasil. Confidence interval resampling source-group, permutation berpasangan dan koreksi Holm disertakan. Semua seed dilaporkan sebagai mean dan SD. McNemar per segmen hanya deskriptif karena segmen sumber yang sama tidak independen.


In [ ]:
def macro_f1_from_counts(counts):
    c=np.asarray(counts,float)
    tn,fp,fn,tp=[c[...,i] for i in range(4)]
    f0=np.divide(2*tn,2*tn+fp+fn,out=np.zeros_like(tn),where=(2*tn+fp+fn)>0)
    f1=np.divide(2*tp,2*tp+fp+fn,out=np.zeros_like(tp),where=(2*tp+fp+fn)>0)
    return (f0+f1)/2

def grouped_confusions(y,pred,groups):
    y=np.asarray(y,int);pred=np.asarray(pred,int);groups=np.asarray(groups,str)
    unique,inverse=np.unique(groups,return_inverse=True)
    cells=2*y+pred
    counts=np.zeros((len(unique),4),int)
    np.add.at(counts,(inverse,cells),1)
    return unique,counts

def paired_group_statistics(frame,model_a,model_b,seed=20261020):
    y=frame.label_final.to_numpy(int);g=frame.group_id.to_numpy(str)
    pa=frame[model_a+'_prediction'].to_numpy(int);pb=frame[model_b+'_prediction'].to_numpy(int)
    unique,ca=grouped_confusions(y,pa,g);_,cb=grouped_confusions(y,pb,g)
    rng=np.random.default_rng(seed);n=len(unique)
    weights=rng.multinomial(n,np.full(n,1/n),size=CONFIG['bootstrap_replicates'])
    differences=macro_f1_from_counts(weights@ca)-macro_f1_from_counts(weights@cb)
    observed=float(macro_f1_from_counts(ca.sum(0))-macro_f1_from_counts(cb.sum(0)))
    swaps=rng.integers(0,2,size=(CONFIG['permutation_replicates'],n))
    diff=ca-cb
    aa=cb.sum(0)+swaps@diff;bb=ca.sum(0)-swaps@diff
    null=macro_f1_from_counts(aa)-macro_f1_from_counts(bb)
    p=float((1+np.sum(np.abs(null)>=abs(observed)-1e-12))/(len(null)+1))
    b=int(np.sum((pa==y)&(pb!=y)));c=int(np.sum((pa!=y)&(pb==y)))
    segment_mcnemar=float(binomtest(b,b+c,.5).pvalue) if b+c else 1.0
    return {'candidate':model_a,'baseline':model_b,'delta_macro_f1':observed,
       'ci_low':float(np.quantile(differences,.025)),'ci_high':float(np.quantile(differences,.975)),
       'group_permutation_p':p,'n_groups':n,'n_rows':len(frame),
       'bootstrap_replicates':len(differences),'permutation_replicates':len(null),
       'segment_mcnemar_descriptive_p':segment_mcnemar,
       'inference_scope':'conditional on stored fitted models; source-group bootstrap/permutation; exploratory internal corpus'}

def holm_adjust(values):
    values=np.asarray(values,float);order=np.argsort(values);result=np.empty(len(values));running=0.
    for rank,i in enumerate(order):
        running=max(running,(len(values)-rank)*values[i]);result[i]=min(1.,running)
    return result

def absolute_group_ci(frame,prediction_column,seed=42):
    u,c=grouped_confusions(frame.label_final,frame[prediction_column],frame.group_id)
    rng=np.random.default_rng(seed)
    w=rng.multinomial(len(u),np.full(len(u),1/len(u)),size=CONFIG['bootstrap_replicates'])
    scores=macro_f1_from_counts(w@c)
    return {'macro_f1_ci_low':float(np.quantile(scores,.025)),'macro_f1_ci_high':float(np.quantile(scores,.975)),
            'clusters':len(u),'bootstrap_replicates':len(scores)}

if CONFIG['run_video']:
    primary=VIDEO_PRED[VIDEO_PRED.seed==CONFIG['primary_seed']].copy()
    assert len(primary)==len(VIDEO)
    comparisons=[('vat_coverage','visual'),('vat_coverage','vat_lr'),('vat_coverage','vat_flags'),
      ('vat_coverage','coverage_only'),('vat_coverage','coverage_shuffled_control')]
    if CONFIG['run_recent_baselines']:comparisons += [('vat_coverage','healnet_adapted_2024'),('vat_coverage','siglip2_vta_probe')]
    paired=pd.DataFrame([paired_group_statistics(primary,a,b,20261020+i) for i,(a,b) in enumerate(comparisons)])
    paired['group_permutation_p_holm']=holm_adjust(paired.group_permutation_p)
    atomic_csv(paired,RUN/'VIDEO_PAIRED_GROUP_STATISTICS.csv')
    cis=[]
    for i,name in enumerate(names):cis.append({'model':name,'seed':CONFIG['primary_seed'],**absolute_group_ci(primary,name+'_prediction',42+i)})
    atomic_csv(pd.DataFrame(cis),RUN/'VIDEO_PRIMARY_F1_CLUSTER_CI.csv')
    slices=[]
    for kind,sets in [('platform',primary.groupby('platform')),
                      ('coverage',primary.groupby(['ocr_ok','stt_ok']))]:
        for key,subset in sets:
            for name in names:
                slices.append({'slice_type':kind,'slice':str(key),'model':name,
                    'groups':subset.group_id.nunique(),**metrics(subset.label_final,subset[name+'_probability'],pred=subset[name+'_prediction'])})
    atomic_csv(pd.DataFrame(slices),RUN/'VIDEO_PLATFORM_AND_COVERAGE_SLICES.csv')
    from sklearn.metrics import roc_curve,precision_recall_curve
    curve_rows=[]
    for name in names:
        fpr,tpr,thr=roc_curve(primary.label_final,primary[name+'_probability'])
        curve_rows.extend({'model':name,'curve':'ROC','x':float(a),'y':float(b)} for a,b in zip(fpr,tpr))
        precision,recall,_=precision_recall_curve(primary.label_final,primary[name+'_probability'])
        curve_rows.extend({'model':name,'curve':'PR','x':float(a),'y':float(b)} for a,b in zip(recall,precision))
    atomic_csv(pd.DataFrame(curve_rows),RUN/'VIDEO_CURVE_COORDINATES.csv')
    display(paired[['candidate','baseline','delta_macro_f1','ci_low','ci_high','group_permutation_p_holm']])

if CONFIG['run_text']:
    tp=pd.concat(TEXT_PREDICTIONS,ignore_index=True)
    base=tp[(tp.model=='indobert_p2')&(tp.seed==42)].copy().sort_values('master_id')
    records=[];abs_ci=[]
    for i,(model,rows) in enumerate(tp[tp.seed==42].groupby('model')):
        rows=rows.sort_values('master_id')
        assert rows.master_id.tolist()==base.master_id.tolist()
        q=base[['master_id','group_id','label_final']].copy().reset_index(drop=True)
        q['indobert_p2_prediction']=base.prediction.to_numpy()
        q[model+'_prediction']=rows.prediction.to_numpy()
        abs_ci.append({'model':model,'seed':42,**absolute_group_ci(q,model+'_prediction',100+i)})
        if model!='indobert_p2':records.append(paired_group_statistics(q,'indobert_p2',model,300+i))
    text_paired=pd.DataFrame(records)
    if len(text_paired):text_paired['group_permutation_p_holm']=holm_adjust(text_paired.group_permutation_p)
    atomic_csv(text_paired,RUN/'TEXT_PAIRED_GROUP_STATISTICS.csv')
    atomic_csv(pd.DataFrame(abs_ci),RUN/'TEXT_PRIMARY_F1_CLUSTER_CI.csv')
    text_slices=[]
    for (model,seed,platform),d in tp.groupby(['model','seed','platform']):
        text_slices.append({'model':model,'seed':seed,'platform':platform,**metrics(d.label_final,d.probability,pred=d.prediction)})
    atomic_csv(pd.DataFrame(text_slices),RUN/'TEXT_PLATFORM_SLICES.csv')



## 10 LOPO sebagai frozen-feature probe

Eksperimen opsional ini memakai fitur pretrained tanpa training label dari held-out platform. Kelompok sumber yang juga muncul pada held-out platform dikeluarkan dari train. Hasilnya tidak boleh dinamai LOPO end-to-end dari model fine-tuned utama.


In [ ]:
# Label-independent pretrained SigLIP2 features only. No global supervised OOF probabilities enter this LOPO probe.
if CONFIG['run_lopo_probe']:
    assert SIGLIP is not None
    X0={'siglip2_visual_probe':SIGLIP['visual'],
        'siglip2_vt_probe':np.column_stack([SIGLIP['visual'],SIGLIP['text']]),
        'siglip2_vta_probe':np.column_stack([SIGLIP['visual'],SIGLIP['text'],AUDIO]),
        'siglip2_vta_coverage_probe':np.column_stack([SIGLIP['visual'],SIGLIP['text'],AUDIO,
            VIDEO.ocr_ok,VIDEO.stt_ok,np.log1p(VIDEO.ocr_length),np.log1p(VIDEO.stt_length)])}
    lopo_metrics=[];lopo_predictions=[]
    for pid,platform in enumerate(sorted(VIDEO.platform.unique())):
        held=np.flatnonzero(VIDEO.platform.to_numpy()==platform)
        held_groups=set(VIDEO.iloc[held].group_id)
        train_pool=np.flatnonzero((VIDEO.platform.to_numpy()!=platform)&~VIDEO.group_id.isin(held_groups).to_numpy())
        assert not(set(VIDEO.iloc[train_pool].group_id)&held_groups)
        fit,cal,rule=choose_group_split(VIDEO,train_pool,.20,900+pid)
        for model,X in X0.items():
            lr=fit_lr(X[fit],VIDEO.iloc[fit].label_final,VIDEO.iloc[fit].group_id.to_numpy(),C=.2)
            pcal=lr.predict_proba(X[cal])[:,1];p=lr.predict_proba(X[held])[:,1]
            threshold=choose_threshold(VIDEO.iloc[cal].label_final,pcal)
            lopo_metrics.append({'held_out_platform':platform,'model':model,'threshold':threshold,
              'training_n':len(fit),'calibration_n':len(cal),'held_out_groups':len(held_groups),
              'protocol':'frozen pretrained feature probe; not end-to-end LOPO of proposed trained model',
              **metrics(VIDEO.iloc[held].label_final,p,threshold=threshold)})
            d=VIDEO.iloc[held][['master_id','platform','group_id','label_final']].copy()
            d['model']=model;d['probability']=p;d['threshold']=threshold;d['prediction']=(p>=threshold).astype(int)
            lopo_predictions.append(d)
        print('LOPO frozen-feature probe complete:',platform)
    atomic_csv(pd.DataFrame(lopo_metrics),RUN/'LOPO_FROZEN_FEATURE_PROBE_METRICS.csv')
    atomic_csv(pd.concat(lopo_predictions,ignore_index=True),RUN/'LOPO_FROZEN_FEATURE_PROBE_PREDICTIONS.csv')



## 11 Ekspor paket hasil untuk pemeriksaan

CSV metrik, prediksi per sampel, manifest, audit alignment, split, ambang, interval, seed summary dan provenance dikemas otomatis. Tidak ada grafik publikasi final yang digambar di notebook ini; pemilihan figure dibahas setelah hasil diketahui.


In [ ]:
METHODS_NOTES = '''# V6 experiment package interpretation

This package is an internal evaluation on an already explored corpus. The corrected text holdout and canonical video folds are not external confirmation.

## Populations and labels
Text: high-agreement social comments, deduplicated after the documented normalization, grouped by informative content URL and platform-scoped author. Every split must contain every source platform. The subset scope must accompany reported scores.
Video: only source rows with matching segment/frame identity and normalized OCR/STT content are eligible. Exclusions are determined before predictions, never by performance. VIDEO_ALIGNMENT_AUDIT.csv retains every original row and eligibility reason. The source master labels are never changed.
LABEL_AUDIT.json checks numerical agreement only. Documentary evidence of human annotation, expert adjudication, task definition, and any child-directed claim remains necessary. GPU computation cannot verify those facts.

## Training and selection
Video uses the same canonical outer groups for all configurations. Each outer training population has four disjoint roles: encoder fit, encoder epoch selection, meta fitting, and decision-threshold calibration. Test groups and identical images are excluded from every training/selection role.
Text and image encoders start from generic pretrained checkpoints and are independently retrained per outer fold and seed. Video text preprocessing is identical during fit and inference. Absent OCR and STT gives a neutral text probability and zero text embedding. Audio standardization/classification fits only the encoder-fit role.
All fusion variants use the same branch probabilities. Meta LR uses C=0.2, group-balanced weights, and meta-fit observations only. Neural fusion uses fixed epochs on the meta-fit role. Main operating thresholds are selected only on calibration. Fixed-0.50 sensitivity is reported for every model.

## Baselines
HEALNet is the official NeurIPS 2024 fusion module pinned by commit and source hash, adapted to fold-specific frozen vector features and a binary head. It is not a reproduction of its biomedical study or original end-to-end setup. A gradient check verifies that each modality participates.
SigLIP2 is the official Google 2025 pretrained vision-language model with frozen features and supervised linear probes on the same outer partitions. This tests a stronger representation baseline and does not isolate the fusion mechanism. SigLIP2 text uses its pretrained 64-token limit.
Coverage-only and coverage-shuffled configurations are negative/shortcut controls. They must not be omitted merely because they perform well.

## Statistical scope
Primary comparisons use seed 42, specified before training. Seeds 2026 and 2027 estimate retraining variability and are all reported; the best test seed is never selected. VIDEO_SEED_SUMMARY.csv contains means and sample standard deviations.
Confidence intervals resample source groups; paired group permutations test predeclared contrasts with Holm correction. Intervals condition on fitted models and do not capture the full uncertainty of new data collection or every training choice. Segment McNemar p values are descriptive because windows within a source are dependent.
Text-extraction dropout is a synthetic fixed-threshold stress test, not a new real-world test set. LOPO outputs are explicitly frozen pretrained-feature probes and must not be presented as LOPO of the full fine-tuned coverage-aware system.

## Figure preparation
Use the stored sample predictions, thresholds, curve coordinates and paired comparisons. No final publication figures are generated here: their selection and design remain a separate discussion. ROC and PR use continuous probabilities; confusion/F1 use stored fold-specific decisions.
Do not label threshold selection as probability calibration. Do not mix this package with legacy 867-segment results or legacy text ensemble scores.
'''
(RUN/'METHODS_AND_CLAIM_BOUNDARIES.md').write_text(METHODS_NOTES)

report_files=[p for p in RUN.rglob('*') if p.is_file()
              and p.suffix.lower() in ('.csv','.json','.md')
              and p.name!='RESULTS_MANIFEST.json']
manifest={'protocol':CONFIG['protocol_version'],'completed_utc':pd.Timestamp.now(tz='UTC').isoformat(),
 'run_directory':str(RUN),'primary_video_seed':CONFIG['primary_seed'],
 'text_n':len(TEXT),'video_n':len(VIDEO),'source_master_sha256':master_hash,
 'notebook_status':'EXECUTED_TO_EXPORT','scope':'internal grouped experiments; no external confirmation',
 'annotation_status':'DOCUMENTARY_REVIEW_REQUIRED',
 'files':[{'name':str(p.relative_to(RUN)),'bytes':p.stat().st_size,'sha256':sha256(p)} for p in sorted(report_files)]}
atomic_json(manifest,RUN/'RESULTS_MANIFEST.json')
archive=RUN/'V6_REVIEWER_RESULTS_PACKAGE.zip';temp=archive.with_suffix('.partial.zip')
with zipfile.ZipFile(temp,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for p in sorted(report_files):z.write(p,str(p.relative_to(RUN)))
    z.write(RUN/'RESULTS_MANIFEST.json','RESULTS_MANIFEST.json')
    for p in (RUN/'vendor').glob('*') if (RUN/'vendor').is_dir() else []:
        if p.is_file() and p not in report_files:z.write(p,'vendor/'+p.name)
temp.replace(archive)
(RUN/'COMPLETE.txt').write_text('V6 REVIEWER EXPERIMENTS COMPLETED\n'+pd.Timestamp.now(tz='UTC').isoformat())
print('SELESAI:',RUN)
print('Paket untuk diperiksa berikutnya:',archive)
print('Gunakan RESULTS_MANIFEST.json dan METHODS_AND_CLAIM_BOUNDARIES.md saat menafsirkan hasil.')

